# 03. Response-Head Training with Edge MLP and GNN

This notebook trains a multi-head response model on the graph artifact created by notebook 02. The prediction target is the four observed response heads:

| Head | Meaning |
|---|---|
| `y_complete` | Whether the user watched at least one full video length. |
| `y_long` | Whether the watch was long by absolute or relative-duration rule. |
| `y_rewatch` | Whether the user watched at least two video lengths. |
| `y_neg` | Whether the watch was very short. |

The model input is one interaction edge `e = (u, v)` where `u` is a user node and `v` is a video node. The notebook supports two model families:

| Model | Uses graph message passing? | Edge features |
|---|---|---|
| `edge_mlp` | No | Uses only the edge feature vector listed below. |
| `gnn` | Yes | Uses graph node embeddings plus the same edge feature vector listed below. |

## Exact Edge Features Used

Notebook 02 saves three edge-feature tensors. The rows of all three tensors are aligned with `edge_index[:, e]`, so row `e` always describes the same user-video interaction.

| Tensor | Number of columns | Treatment in this notebook |
|---|---:|---|
| `x_cont` | 23 | Standardized using train rows only, then concatenated directly. |
| `x_bin` | 6 | Stored as 0/1 indicators, then concatenated directly. |
| `x_cat` | 35 | Stored as integer category codes with `0 = UNK`, then embedded. |

These are the exact columns in `x_cont`:

| Feature | Description |
|---|---|
| `u_follow_user_num` | User profile: number of followed users. |
| `u_fans_user_num` | User profile: number of fans/followers. |
| `u_friend_user_num` | User profile: number of friends. |
| `u_register_days` | User profile: days since registration. |
| `u_follow_user_num_log1p` | `log(1 + u_follow_user_num)`, included to reduce scale skew. |
| `u_fans_user_num_log1p` | `log(1 + u_fans_user_num)`, included to reduce scale skew. |
| `u_friend_user_num_log1p` | `log(1 + u_friend_user_num)`, included to reduce scale skew. |
| `u_register_days_log1p` | `log(1 + u_register_days)`, included to reduce scale skew. |
| `i_aspect_ratio` | Item metadata: video aspect ratio. |
| `i_video_duration` | Item metadata: video duration. |
| `i_age_since_upload_days` | Item-context feature: days between upload time and interaction time. |
| `ctx_hour_sin` | Time-context feature: sine encoding of hour of day. |
| `ctx_hour_cos` | Time-context feature: cosine encoding of hour of day. |
| `hist_ema_y_complete` | User history before this row: EMA of previous `y_complete`. |
| `hist_ema_y_long` | User history before this row: EMA of previous `y_long`. |
| `hist_ema_y_rewatch` | User history before this row: EMA of previous `y_rewatch`. |
| `hist_ema_y_neg` | User history before this row: EMA of previous `y_neg`. |
| `hist_ema_watchratio` | User history before this row: EMA of previous watch ratio. |
| `hist_cat_ema_complete` | User-category history before this row: EMA completion rate for the current level-1 category. |
| `hist_cat_entropy_l2` | User history before this row: entropy of the user's historical level-1 category distribution, normalized by `log(number of categories)`. |
| `hist_author_recency_days` | User-author history before this row: days since this user last saw this author. |
| `hist_prev_sess_len` | User history before this row: length of the previous session. |
| `hist_intersess_gap_h` | User history before this row: hours since the previous session. |

These are the exact columns in `x_bin`:

| Feature | Description |
|---|---|
| `u_is_lowactive_period` | User profile flag for low activity period. |
| `u_is_live_streamer` | User profile flag for whether the user is a live streamer. |
| `u_is_video_author` | User profile flag for whether the user is a video author. |
| `ctx_is_weekend` | Time-context flag for weekend interaction. |
| `hist_last_complete_author` | User-author history before this row: whether the last exposure to this author was completed. |
| `hist_has_author_history` | User-author history before this row: whether this user has previously seen this author. |

These are the exact columns in `x_cat`:

| Feature | Description |
|---|---|
| `burst_id` | Burst identifier encoded as a category, not as a number with distance meaning. |
| `session` | Session identifier encoded as a category, not as a continuous/order feature. |
| `u_user_active_degree` | User profile active-degree bucket. |
| `u_follow_user_num_range` | User profile bucket for followed-user count. |
| `u_fans_user_num_range` | User profile bucket for fan/follower count. |
| `u_friend_user_num_range` | User profile bucket for friend count. |
| `u_register_days_range` | User profile bucket for registration age. |
| `u_onehot_feat0` | User profile categorical one-hot field 0, stored as a categorical code. |
| `u_onehot_feat1` | User profile categorical one-hot field 1, stored as a categorical code. |
| `u_onehot_feat2` | User profile categorical one-hot field 2, stored as a categorical code. |
| `u_onehot_feat3` | User profile categorical one-hot field 3, stored as a categorical code. |
| `u_onehot_feat4` | User profile categorical one-hot field 4, stored as a categorical code. |
| `u_onehot_feat5` | User profile categorical one-hot field 5, stored as a categorical code. |
| `u_onehot_feat6` | User profile categorical one-hot field 6, stored as a categorical code. |
| `u_onehot_feat7` | User profile categorical one-hot field 7, stored as a categorical code. |
| `u_onehot_feat8` | User profile categorical one-hot field 8, stored as a categorical code. |
| `u_onehot_feat9` | User profile categorical one-hot field 9, stored as a categorical code. |
| `u_onehot_feat10` | User profile categorical one-hot field 10, stored as a categorical code. |
| `u_onehot_feat11` | User profile categorical one-hot field 11, stored as a categorical code. |
| `u_onehot_feat12` | User profile categorical one-hot field 12, stored as a categorical code. |
| `u_onehot_feat13` | User profile categorical one-hot field 13, stored as a categorical code. |
| `u_onehot_feat14` | User profile categorical one-hot field 14, stored as a categorical code. |
| `u_onehot_feat15` | User profile categorical one-hot field 15, stored as a categorical code. |
| `u_onehot_feat16` | User profile categorical one-hot field 16, stored as a categorical code. |
| `u_onehot_feat17` | User profile categorical one-hot field 17, stored as a categorical code. |
| `i_author_id` | Item metadata: video author ID. |
| `i_video_type` | Item metadata: video type. |
| `i_upload_type` | Item metadata: upload type. |
| `i_visible_status` | Item metadata: visibility status. |
| `i_music_id` | Item metadata: music ID. |
| `i_video_tag_id` | Item metadata: video tag ID. |
| `i_video_tag_name` | Item metadata: video tag name. |
| `i_cat_level1_id` | Item metadata: level-1 category ID. |
| `i_cat_level2_id` | Item metadata: level-2 category ID. |
| `i_cat_level3_id` | Item metadata: level-3 category ID. |

The graph side uses IDs differently. `user_id` and `video_id` are not columns inside `x_cont`, `x_bin`, or `x_cat`; they define the endpoints in `edge_index`. In the GNN model, those endpoints select learned node embeddings and receive message passing from allowed historical/context edges. In the EdgeMLP model, the endpoints are ignored except through the edge features above.

The graph side is deliberately split-specific:

| Split being scored | Message-passing graph | Query edges |
|---|---|---|
| Train | `edge_index_train_mp` | `edge_index[:, train_idx]` |
| Validation | `edge_index_val_mp` | `edge_index[:, val_idx]` |
| Test | `edge_index_test_mp` | `edge_index[:, test_idx]` |

Validation message passing uses train sessions only. Test message passing uses train + validation sessions by default, never test sessions. This keeps future exposure edges out of the node embeddings used for validation/test scoring.

For every edge, the model outputs four logits:

```text
logits_e = [logit_complete, logit_long, logit_rewatch, logit_neg]
```

and optimizes `BCEWithLogitsLoss` over all four heads.


## Table of Contents

1. Configure runtime and load the clean graph artifact.
2. Validate feature tensors, label tensors, split indices, and message-passing graphs.
3. Encode edge features, especially categorical IDs.
4. Define the edge-only MLP baseline.
5. Define the leakage-safe GNN.
6. Train with validation-based model selection.
7. Evaluate on the test split.
8. Save checkpoint, history, and metrics.


## 0. Setup

Main switches:

| Variable | Meaning |
|---|---|
| `USE_TINY` | If `True`, load `gnn_data_tiny.pt` for smoke tests. If `False`, load full `gnn_data.pt`. |
| `MODEL_TYPE` | `"edge_mlp"` or `"gnn"` in the training cell. |
| `SAVE_OUTPUTS` | If `True`, save checkpoint, training history, and metrics. |

Full-run checklist:

1. Rerun notebook 02 with `WRITE_OUTPUTS = True`.
2. Set `USE_TINY = False`.
3. Rerun this setup cell and the load cell.
4. Confirm the load cell prints `gnn_data.pt`.
5. Train and save.

If `torch_geometric` is unavailable, the notebook falls back to a pure-PyTorch sparse GCN layer. The fallback is useful for smoke tests; PyG is recommended for full GNN training speed.


In [ ]:
from pathlib import Path
import copy
import math
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

try:
    from torch_geometric.nn import GCNConv
    GCN_LAYER_CLS = GCNConv
    USING_TORCH_GEOMETRIC = True
except ImportError:
    USING_TORCH_GEOMETRIC = False

    class SparseGCNConv(nn.Module):
        """Small pure-PyTorch fallback for GCNConv.

        It computes:

            H' = D^{-1/2} (A + I) D^{-1/2} H W

        using a sparse COO adjacency. This is here so the notebook can run
        in environments without torch_geometric. PyG is still preferred for
        full-data training speed.
        """

        def __init__(self, in_channels, out_channels):
            super().__init__()
            self.linear = nn.Linear(in_channels, out_channels, bias=False)
            self.bias = nn.Parameter(torch.zeros(out_channels))

        def forward(self, x, edge_index):
            num_nodes = x.size(0)
            row, col = edge_index[0], edge_index[1]

            loop = torch.arange(num_nodes, device=x.device, dtype=row.dtype)
            row = torch.cat([row, loop])
            col = torch.cat([col, loop])

            deg = torch.bincount(col, minlength=num_nodes).float()
            deg_inv_sqrt = deg.clamp(min=1).pow(-0.5)
            values = deg_inv_sqrt[row] * deg_inv_sqrt[col]

            adj = torch.sparse_coo_tensor(
                torch.stack([col, row]),
                values,
                (num_nodes, num_nodes),
                device=x.device,
            ).coalesce()
            out = torch.sparse.mm(adj, x)
            return self.linear(out) + self.bias

    GCN_LAYER_CLS = SparseGCNConv
    warnings.warn(
        "torch_geometric is not installed; using a pure-PyTorch SparseGCNConv fallback. "
        "This is fine for smoke tests, but PyG is recommended for full-data training."
    )

PROJECT_ROOT = Path("/Users/haozhangao/Desktop/RecSys Research")
BASE = PROJECT_ROOT / "KuaiRec 2.0" / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "python_code_new" / "model_outputs"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

USE_TINY = False
DATA_PATH = BASE / ("gnn_data_tiny.pt" if USE_TINY else "gnn_data.pt")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RANDOM_SEED = 20260613

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("data path:", DATA_PATH)
print("device:", DEVICE)


data path: /Users/haozhangao/Desktop/RecSys Research/KuaiRec 2.0/data/processed/gnn_data_tiny.pt
device: cpu


/var/folders/kx/mcwvbnt12wl9fy5xcs5jlnx00000gn/T/ipykernel_42874/2272413872.py:59: UserWarning: torch_geometric is not installed; using a pure-PyTorch SparseGCNConv fallback. This is fine for smoke tests, but PyG is recommended for full-data training.
  warnings.warn(


## 1. Load and Validate Graph Artifact

The artifact is a dictionary saved by notebook 02. It must contain:

| Key | Shape | Role |
|---|---:|---|
| `edge_index` | `[2, E]` | Directed query edges to score. |
| `edge_index_train_mp` | `[2, 2E_train_context]` | Undirected train message-passing graph. |
| `edge_index_val_mp` | `[2, 2E_val_context]` | Undirected validation context graph. |
| `edge_index_test_mp` | `[2, 2E_test_context]` | Undirected test context graph. |
| `x_cont` | `[E, D_cont]` | Standardized continuous features. |
| `x_bin` | `[E, D_bin]` | Binary features. |
| `x_cat` | `[E, D_cat]` | Categorical indices for embeddings. |
| `y` | `[E, 4]` | Multi-head labels. |
| `train_idx`, `val_idx`, `test_idx` | `[E_split]` | Query edge indices for each split. |
| `feature_metadata` | dictionary | Feature names, category cardinalities, and scaling metadata. |

The load cell checks that these shapes line up. It also checks `USE_TINY` and `DATA_PATH` agree, because it is easy in notebooks to edit `USE_TINY` but forget to rerun the setup cell.

PyTorch 2.6 changed `torch.load` to default to `weights_only=True`; this data artifact is a trusted local Python object, so the notebook loads it with `weights_only=False`.


In [2]:
expected_data_name = "gnn_data_tiny.pt" if USE_TINY else "gnn_data.pt"
if DATA_PATH.name != expected_data_name:
    raise ValueError(
        f"USE_TINY={USE_TINY} implies {expected_data_name}, but DATA_PATH is {DATA_PATH}. "
        "Rerun the setup cell before loading data."
    )

print("USE_TINY:", USE_TINY)
print("loading data from:", DATA_PATH)
# PyTorch 2.6 defaults to weights_only=True. Our trusted local data artifact
# contains metadata dictionaries with pandas-derived objects, so load it as a
# regular Python object.
data = torch.load(DATA_PATH, map_location="cpu", weights_only=False)

required_keys = [
    "edge_index",
    "edge_index_train_mp",
    "edge_index_val_mp",
    "edge_index_test_mp",
    "x_cont",
    "x_bin",
    "x_cat",
    "y",
    "train_idx",
    "val_idx",
    "test_idx",
    "num_nodes",
    "feature_metadata",
    "test_context_policy",
]
missing = [k for k in required_keys if k not in data]
if missing:
    raise KeyError(
        "This is not the clean notebook-02 artifact. Missing keys: "
        f"{missing}. Rerun notebook 02 after the session-level split rewrite."
    )

if "edge_attr" in data:
    warnings.warn("Artifact still contains legacy edge_attr; this notebook ignores it.")

metadata = data["feature_metadata"]
continuous_cols = metadata["continuous_cols"]
binary_cols = metadata["binary_cols"]
categorical_cols = metadata["categorical_cols"]
categorical_cardinalities = metadata["categorical_cardinalities"]
label_names = data.get("label_names", ["complete", "long", "rewatch", "negative"])

checks = {
    "edge_index_shape": data["edge_index"].ndim == 2 and data["edge_index"].shape[0] == 2,
    "train_mp_shape": data["edge_index_train_mp"].ndim == 2 and data["edge_index_train_mp"].shape[0] == 2,
    "val_mp_shape": data["edge_index_val_mp"].ndim == 2 and data["edge_index_val_mp"].shape[0] == 2,
    "test_mp_shape": data["edge_index_test_mp"].ndim == 2 and data["edge_index_test_mp"].shape[0] == 2,
    "x_cont_rows": data["x_cont"].shape[0] == data["edge_index"].shape[1],
    "x_bin_rows": data["x_bin"].shape[0] == data["edge_index"].shape[1],
    "x_cat_rows": data["x_cat"].shape[0] == data["edge_index"].shape[1],
    "y_shape": data["y"].shape == (data["edge_index"].shape[1], 4),
    "categorical_width": data["x_cat"].shape[1] == len(categorical_cols),
    "cardinality_width": len(categorical_cardinalities) == len(categorical_cols),
}
for name, ok in checks.items():
    print(f"{name}: {ok}")
failed = [name for name, ok in checks.items() if not ok]
if failed:
    raise ValueError(f"Artifact validation failed: {failed}")

print("test_context_policy:", data["test_context_policy"])
print("edges:", data["edge_index"].shape[1])
print("num_nodes:", data["num_nodes"])
print("feature dims:", {
    "continuous": data["x_cont"].shape[1],
    "binary": data["x_bin"].shape[1],
    "categorical": data["x_cat"].shape[1],
})
print("split sizes:", {
    "train": data["train_idx"].numel(),
    "val": data["val_idx"].numel(),
    "test": data["test_idx"].numel(),
})


edge_index_shape: True
train_mp_shape: True
val_mp_shape: True
test_mp_shape: True
x_cont_rows: True
x_bin_rows: True
x_cat_rows: True
y_shape: True
categorical_width: True
cardinality_width: True
test_context_policy: train_val
edges: 615575
num_nodes: 17904
feature dims: {'continuous': 23, 'binary': 6, 'categorical': 35}
split sizes: {'train': 545484, 'val': 45783, 'test': 24308}


In [3]:
def move_to_device(data, device):
    tensor_keys = [
        "edge_index",
        "edge_index_train_mp",
        "edge_index_val_mp",
        "edge_index_test_mp",
        "x_cont",
        "x_bin",
        "x_cat",
        "y",
        "train_idx",
        "val_idx",
        "test_idx",
    ]
    moved = dict(data)
    for key in tensor_keys:
        moved[key] = moved[key].to(device)
    return moved

data = move_to_device(data, DEVICE)

display(pd.DataFrame({
    "group": ["continuous", "binary", "categorical"],
    "n_features": [len(continuous_cols), len(binary_cols), len(categorical_cols)],
}))
display(pd.Series(categorical_cardinalities, index=categorical_cols, name="cardinality").sort_values(ascending=False).to_frame().head(20))


,group,n_features
0,continuous,23
1,binary,6
2,categorical,35


,cardinality
i_music_id,7664
i_author_id,7408
u_onehot_feat3,1077
i_video_tag_id,547
i_video_tag_name,511
u_onehot_feat8,341
session,241
i_cat_level3_id,215
i_cat_level2_id,138
u_onehot_feat7,48


## 2. Edge Feature Encoder

The edge feature encoder builds the non-graph part of the edge representation using exactly the 64 edge columns listed in the opening feature tables: 23 continuous features, 6 binary features, and 35 categorical features.

For edge `e`, define:

```text
q_e = concat(x_cont_e, x_bin_e, emb_1[x_cat_e1], ..., emb_35[x_cat_e35])
```

where each categorical feature has its own embedding table.

Important details:

- `x_cat = 0` means unknown or unseen category.
- Train-seen categories start at index 1.
- Category IDs such as `i_author_id`, `i_music_id`, and `i_cat_level1_id` are not treated as continuous numbers.
- `user_id` and `video_id` are not edge-feature columns; they are used only through `edge_index` for graph endpoints.
- Embedding dimension is chosen by:

```text
dim_c = min(32, max(2, round(cardinality_c ** 0.25 * 4)))
```

This keeps high-cardinality variables compact while still letting the model learn category-specific effects.


In [4]:
def default_embedding_dim(cardinality):
    return min(32, max(2, int(round(cardinality ** 0.25 * 4))))


class EdgeFeatureEncoder(nn.Module):
    def __init__(self, categorical_cardinalities, continuous_dim, binary_dim, embedding_dims=None):
        super().__init__()
        self.continuous_dim = int(continuous_dim)
        self.binary_dim = int(binary_dim)
        self.categorical_cardinalities = [int(c) for c in categorical_cardinalities]

        if embedding_dims is None:
            embedding_dims = [default_embedding_dim(c) for c in self.categorical_cardinalities]
        if len(embedding_dims) != len(self.categorical_cardinalities):
            raise ValueError("embedding_dims must match categorical_cardinalities")

        self.embedding_dims = [int(d) for d in embedding_dims]
        self.embeddings = nn.ModuleList([
            nn.Embedding(cardinality, dim)
            for cardinality, dim in zip(self.categorical_cardinalities, self.embedding_dims)
        ])

        self.output_dim = self.continuous_dim + self.binary_dim + sum(self.embedding_dims)

    def forward(self, x_cont, x_bin, x_cat):
        parts = []
        if self.continuous_dim:
            parts.append(x_cont.float())
        if self.binary_dim:
            parts.append(x_bin.float())

        if self.embeddings:
            cat_parts = []
            for j, emb in enumerate(self.embeddings):
                codes = x_cat[:, j].long().clamp(min=0, max=emb.num_embeddings - 1)
                cat_parts.append(emb(codes))
            parts.extend(cat_parts)

        return torch.cat(parts, dim=-1)


encoder_preview = EdgeFeatureEncoder(
    categorical_cardinalities=categorical_cardinalities,
    continuous_dim=len(continuous_cols),
    binary_dim=len(binary_cols),
)
print("edge encoder output dim:", encoder_preview.output_dim)
print("first 10 embedding dims:", encoder_preview.embedding_dims[:10])
del encoder_preview


edge encoder output dim: 392
first 10 embedding dims: [6, 16, 6, 7, 7, 7, 7, 5, 7, 9]


## 3. Edge-Only MLP Baseline

The edge-only MLP uses only `q_e`, the encoded edge features:

```text
logits_e = MLP(q_e)
```

It does not use node embeddings or graph message passing. This is a strong baseline because notebook 01 already built rich history features, user features, item features, and category features.

If this model beats the GNN, that means the engineered edge features are doing most of the predictive work, or the exposure graph is not adding useful signal under the leakage-safe evaluation.


In [5]:
class EdgeMLP(nn.Module):
    def __init__(self, categorical_cardinalities, continuous_dim, binary_dim, hidden_dim=128, dropout=0.0, num_heads=4):
        super().__init__()
        self.edge_encoder = EdgeFeatureEncoder(categorical_cardinalities, continuous_dim, binary_dim)
        self.net = nn.Sequential(
            nn.Linear(self.edge_encoder.output_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_heads),
        )

    def encode_nodes(self, message_edge_index):
        return None

    def score_edges(self, node_emb, query_edge_index, x_cont, x_bin, x_cat):
        edge_repr = self.edge_encoder(x_cont, x_bin, x_cat)
        return self.net(edge_repr)


## 4. Leakage-Safe GNN

The GNN learns a base embedding for every user and item node, then applies GCN message passing over the allowed context graph.

For split `s`, with message-passing graph `A_s`:

```text
H_s^(0) = learned node embeddings
H_s^(l+1) = ReLU(GCN(A_s, H_s^(l)))
```

For query edge `e = (u, v)`, the edge score uses:

```text
g_e = concat(H_s[u], H_s[v], q_e)
logits_e = MLP(g_e)
```

The split matters:

- train scoring uses node embeddings encoded from `edge_index_train_mp`;
- validation scoring uses node embeddings encoded from `edge_index_val_mp`;
- test scoring uses node embeddings encoded from `edge_index_test_mp`.

Validation/test query edges are allowed as edge pairs to score, but their edge events are not included in the message-passing graph for that evaluation period.


In [6]:
class CleanGNN(nn.Module):
    def __init__(
        self,
        num_nodes,
        categorical_cardinalities,
        continuous_dim,
        binary_dim,
        hidden_dim=64,
        num_heads=4,
        num_gcn_layers=3,
        dropout=0.0,
    ):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, hidden_dim)
        self.gcn_layers = nn.ModuleList([
            GCN_LAYER_CLS(hidden_dim, hidden_dim)
            for _ in range(num_gcn_layers)
        ])
        self.dropout = float(dropout)
        self.edge_encoder = EdgeFeatureEncoder(categorical_cardinalities, continuous_dim, binary_dim)

        input_dim = 2 * hidden_dim + self.edge_encoder.output_dim
        self.edge_mlp = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_heads),
        )

    def encode_nodes(self, message_edge_index):
        x = self.node_emb.weight
        for layer in self.gcn_layers:
            x = layer(x, message_edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return x

    def score_edges(self, node_emb, query_edge_index, x_cont, x_bin, x_cat):
        src, dst = query_edge_index
        edge_repr = self.edge_encoder(x_cont, x_bin, x_cat)
        full_repr = torch.cat([node_emb[src], node_emb[dst], edge_repr], dim=-1)
        return self.edge_mlp(full_repr)


## 5. Training and Evaluation Utilities

The training loop uses mini-batch stochastic optimization over query edges.

Current convention:

```text
one loop iteration = one sampled mini-batch update
```

The variable is named `epoch` in the code, but technically it is closer to a training step because the code samples one random mini-batch per iteration instead of sweeping through all train edges once.

For each training step:

1. Encode train node embeddings from `edge_index_train_mp`.
2. Sample `BATCH_SIZE` train query edges.
3. Score those edges.
4. Compute four-head BCE loss.
5. Backpropagate and update parameters.

For evaluation:

- encode nodes from the split-specific message graph;
- score all query edges in that split in batches;
- compute average BCE loss;
- compute ROC AUC per head and mean AUC across valid heads.


In [7]:
def make_model(model_type="gnn", hidden_dim=64, dropout=0.0):
    kwargs = dict(
        categorical_cardinalities=categorical_cardinalities,
        continuous_dim=len(continuous_cols),
        binary_dim=len(binary_cols),
    )
    if model_type == "gnn":
        return CleanGNN(
            num_nodes=int(data["num_nodes"]),
            hidden_dim=hidden_dim,
            dropout=dropout,
            **kwargs,
        ).to(DEVICE)
    if model_type == "edge_mlp":
        return EdgeMLP(
            hidden_dim=128,
            dropout=dropout,
            **kwargs,
        ).to(DEVICE)
    raise ValueError("model_type must be 'gnn' or 'edge_mlp'")


def split_idx(split):
    return {
        "train": data["train_idx"],
        "val": data["val_idx"],
        "test": data["test_idx"],
    }[split]


def split_message_graph(split):
    return {
        "train": data["edge_index_train_mp"],
        "val": data["edge_index_val_mp"],
        "test": data["edge_index_test_mp"],
    }[split]


def score_edge_ids(model, node_emb, edge_ids):
    query_edges = data["edge_index"][:, edge_ids]
    return model.score_edges(
        node_emb,
        query_edges,
        data["x_cont"][edge_ids],
        data["x_bin"][edge_ids],
        data["x_cat"][edge_ids],
    )


def encode_for_split(model, split):
    message_graph = split_message_graph(split)
    return model.encode_nodes(message_graph)


criterion = nn.BCEWithLogitsLoss()


def train_step(model, optimizer, batch_size=4096):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    node_emb = encode_for_split(model, "train")
    train_edges = data["train_idx"]
    batch_pos = torch.randint(0, train_edges.numel(), (batch_size,), device=DEVICE)
    edge_ids = train_edges[batch_pos]

    logits = score_edge_ids(model, node_emb, edge_ids)
    loss = criterion(logits, data["y"][edge_ids])
    loss.backward()
    optimizer.step()
    return float(loss.item())


@torch.no_grad()
def evaluate_split(model, split, batch_size=50000):
    model.eval()
    edge_ids_all = split_idx(split)
    node_emb = encode_for_split(model, split)

    losses = []
    probs = []
    labels = []
    n_total = 0

    for start in range(0, edge_ids_all.numel(), batch_size):
        edge_ids = edge_ids_all[start:start + batch_size]
        logits = score_edge_ids(model, node_emb, edge_ids)
        y_batch = data["y"][edge_ids]
        loss = criterion(logits, y_batch)

        n = int(edge_ids.numel())
        losses.append(float(loss.item()) * n)
        n_total += n
        probs.append(torch.sigmoid(logits).detach().cpu().numpy())
        labels.append(y_batch.detach().cpu().numpy())

    y_true = np.concatenate(labels, axis=0)
    y_prob = np.concatenate(probs, axis=0)
    avg_loss = sum(losses) / max(n_total, 1)

    aucs = []
    for h in range(y_true.shape[1]):
        y_h = y_true[:, h]
        if np.all(y_h == 0) or np.all(y_h == 1):
            aucs.append(float("nan"))
        else:
            aucs.append(float(roc_auc_score(y_h, y_prob[:, h])))

    valid_aucs = [a for a in aucs if not np.isnan(a)]
    mean_auc = float(np.mean(valid_aucs)) if valid_aucs else float("nan")
    return avg_loss, aucs, mean_auc


## 6. Train Loop and Model Selection

The helper `train_model(...)` trains for a fixed number of iterations and tracks validation mean AUC.

It performs **model selection** but not true early stopping:

- after each evaluation, if validation mean AUC improves, the notebook copies the current `state_dict`;
- training still continues until `num_epochs` is reached;
- after the loop, the best-validation checkpoint is restored;
- test metrics are computed once using that restored checkpoint.

To do a smoke test, use `USE_TINY = True` and a small `NUM_EPOCHS`.

For a full EdgeMLP run, a typical block is:

```python
MODEL_TYPE = "edge_mlp"
NUM_EPOCHS = 2000
BATCH_SIZE = 4096
LR = 1e-3
HIDDEN_DIM = 32
DROPOUT = 0.0
```

For a full GNN run, use `MODEL_TYPE = "gnn"` and tune `HIDDEN_DIM`, learning rate, and number of GCN layers if needed.


In [8]:
def train_model(
    model_type="gnn",
    hidden_dim=64,
    dropout=0.0,
    lr=1e-3,
    batch_size=4096,
    num_epochs=5,
    eval_every=1,
):
    model = make_model(model_type=model_type, hidden_dim=hidden_dim, dropout=dropout)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    best_state = None
    best_val_auc = -np.inf

    for epoch in range(1, num_epochs + 1):
        step_loss = train_step(model, optimizer, batch_size=batch_size)

        if epoch == 1 or epoch % eval_every == 0:
            train_loss, train_auc_heads, train_auc = evaluate_split(model, "train")
            val_loss, val_auc_heads, val_auc = evaluate_split(model, "val")

            row = {
                "epoch": epoch,
                "step_loss": step_loss,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "train_auc_mean": train_auc,
                "val_auc_mean": val_auc,
            }
            for name, auc in zip(label_names, val_auc_heads):
                row[f"val_auc_{name}"] = auc
            history.append(row)

            print(
                f"epoch={epoch:03d} "
                f"step_loss={step_loss:.4f} "
                f"train_auc={train_auc:.4f} "
                f"val_auc={val_auc:.4f}"
            )

            if val_auc > best_val_auc:
                best_val_auc = val_auc
                best_state = copy.deepcopy(model.state_dict())

    if best_state is not None:
        model.load_state_dict(best_state)

    test_loss, test_auc_heads, test_auc = evaluate_split(model, "test")
    print("best val_auc:", best_val_auc)
    print("test_loss:", test_loss)
    print("test_auc_mean:", test_auc)
    print("test_auc_heads:", dict(zip(label_names, test_auc_heads)))

    return model, pd.DataFrame(history), {
        "best_val_auc": float(best_val_auc),
        "test_loss": float(test_loss),
        "test_auc_mean": float(test_auc),
        "test_auc_heads": dict(zip(label_names, [float(x) for x in test_auc_heads])),
    }


In [9]:
# Smoke-test defaults. Increase epochs for the real run.
MODEL_TYPE = "gnn"  # choices: "gnn", "edge_mlp"
NUM_EPOCHS = 5 if USE_TINY else 50
BATCH_SIZE = 4096
LR = 1e-3
HIDDEN_DIM = 64
DROPOUT = 0.0

model, history, final_metrics = train_model(
    model_type=MODEL_TYPE,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
    lr=LR,
    batch_size=BATCH_SIZE,
    num_epochs=NUM_EPOCHS,
    eval_every=1 if USE_TINY else 5,
)

display(history.tail())
final_metrics


/var/folders/kx/mcwvbnt12wl9fy5xcs5jlnx00000gn/T/ipykernel_42874/2272413872.py:49: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:767.)
  adj = torch.sparse_coo_tensor(


epoch=001 step_loss=0.6899 train_auc=0.5275 val_auc=0.5338
epoch=002 step_loss=0.6775 train_auc=0.5296 val_auc=0.5340
epoch=003 step_loss=0.6641 train_auc=0.5304 val_auc=0.5317
epoch=004 step_loss=0.6496 train_auc=0.5296 val_auc=0.5276
epoch=005 step_loss=0.6315 train_auc=0.5277 val_auc=0.5231
best val_auc: 0.5339689661347425
test_loss: 0.665984570980072
test_auc_mean: 0.5454660557530626
test_auc_heads: {'y_complete': 0.5377531161334707, 'y_long': 0.5500635052048672, 'y_rewatch': 0.5594597708553287, 'y_neg': 0.5345878308185839}


,epoch,step_loss,train_loss,val_loss,train_auc_mean,val_auc_mean,val_auc_y_complete,val_auc_y_long,val_auc_y_rewatch,val_auc_y_neg
0,1,0.689866,0.677243,0.678086,0.527513,0.533784,0.559106,0.491240,0.585261,0.499529
1,2,0.677547,0.664501,0.665302,0.529587,0.533969,0.553718,0.505788,0.569043,0.507327
2,3,0.664127,0.649553,0.650380,0.530415,0.531652,0.544000,0.517455,0.552215,0.512938
3,4,0.649565,0.631080,0.631967,0.529628,0.527598,0.533561,0.522662,0.533163,0.521006
4,5,0.631528,0.608456,0.609417,0.527739,0.523079,0.521774,0.524126,0.517269,0.529147


{'best_val_auc': 0.5339689661347425,
 'test_loss': 0.665984570980072,
 'test_auc_mean': 0.5454660557530626,
 'test_auc_heads': {'y_complete': 0.5377531161334707,
  'y_long': 0.5500635052048672,
  'y_rewatch': 0.5594597708553287,
  'y_neg': 0.5345878308185839}}

In [18]:
# Full-data training run
MODEL_TYPE = "edge_mlp"  # choices: "gnn", "edge_mlp"

NUM_EPOCHS = 2000
BATCH_SIZE = 4096
LR = 1e-3
HIDDEN_DIM = 32
DROPOUT = 0.0

model, history, final_metrics = train_model(
    model_type=MODEL_TYPE,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
    lr=LR,
    batch_size=BATCH_SIZE,
    num_epochs=NUM_EPOCHS,
    eval_every=5,
)

display(history.tail())
final_metrics

epoch=001 step_loss=0.6842 train_auc=0.5027 val_auc=0.5011
epoch=005 step_loss=0.6382 train_auc=0.5218 val_auc=0.5184
epoch=010 step_loss=0.5230 train_auc=0.5270 val_auc=0.5164
epoch=015 step_loss=0.4688 train_auc=0.5437 val_auc=0.5377
epoch=020 step_loss=0.4727 train_auc=0.5941 val_auc=0.5980
epoch=025 step_loss=0.4413 train_auc=0.6267 val_auc=0.6327
epoch=030 step_loss=0.4463 train_auc=0.6542 val_auc=0.6600
epoch=035 step_loss=0.4243 train_auc=0.6761 val_auc=0.6817
epoch=040 step_loss=0.4283 train_auc=0.6896 val_auc=0.6950
epoch=045 step_loss=0.4235 train_auc=0.7017 val_auc=0.7075
epoch=050 step_loss=0.4092 train_auc=0.7169 val_auc=0.7233
epoch=055 step_loss=0.4007 train_auc=0.7301 val_auc=0.7380
epoch=060 step_loss=0.3964 train_auc=0.7386 val_auc=0.7477
epoch=065 step_loss=0.3970 train_auc=0.7447 val_auc=0.7553
epoch=070 step_loss=0.3906 train_auc=0.7489 val_auc=0.7598
epoch=075 step_loss=0.3948 train_auc=0.7537 val_auc=0.7644
epoch=080 step_loss=0.3888 train_auc=0.7582 val_auc=0.76

,epoch,step_loss,train_loss,val_loss,train_auc_mean,val_auc_mean,val_auc_y_complete,val_auc_y_long,val_auc_y_rewatch,val_auc_y_neg
396,1980,0.295962,0.297431,0.343163,0.880056,0.834563,0.844663,0.807032,0.827802,0.858753
397,1985,0.295733,0.296351,0.342515,0.880123,0.835272,0.845521,0.807267,0.827475,0.860823
398,1990,0.292474,0.295554,0.340551,0.880537,0.834491,0.845534,0.806328,0.824859,0.861242
399,1995,0.309324,0.295995,0.342936,0.880147,0.834595,0.843890,0.807998,0.826673,0.859817
400,2000,0.292781,0.295504,0.340743,0.880685,0.835690,0.845648,0.810098,0.827326,0.859686


{'best_val_auc': 0.8460264679688592,
 'test_loss': 0.3596462607383728,
 'test_auc_mean': 0.8141119163084312,
 'test_auc_heads': {'y_complete': 0.8168825839621238,
  'y_long': 0.7866485786425981,
  'y_rewatch': 0.7992755651298977,
  'y_neg': 0.8536409374991052}}

## 7. Save Model and Metrics

Saved files go to:

```text
python_code_new/model_outputs/
```

The save cell writes three files whose names are based on the actual model object:

| Actual model | Checkpoint | History | Metrics |
|---|---|---|---|
| `EdgeMLP` | `edge_mlp_{tiny/full}.pt` | `edge_mlp_{tiny/full}_history.csv` | `edge_mlp_{tiny/full}_metrics.json` |
| `CleanGNN` | `gnn_{tiny/full}.pt` | `gnn_{tiny/full}_history.csv` | `gnn_{tiny/full}_metrics.json` |

The important point is that the saved filename follows the trained model object, not a stale notebook variable. If you trained an EdgeMLP but forgot to change `MODEL_TYPE`, the checkpoint is still saved under `edge_mlp_*`.


In [ ]:
SAVE_OUTPUTS = True

if SAVE_OUTPUTS:
    suffix = "tiny" if DATA_PATH.name.endswith("_tiny.pt") else "full"
    if (suffix == "tiny") != bool(USE_TINY):
        raise ValueError("USE_TINY and DATA_PATH disagree. Rerun the setup/load cells before saving.")

    if isinstance(model, EdgeMLP):
        actual_model_type = "edge_mlp"
    elif isinstance(model, CleanGNN):
        actual_model_type = "gnn"
    else:
        raise TypeError(f"Unexpected model class: {type(model).__name__}")

    if actual_model_type != MODEL_TYPE:
        warnings.warn(
            f"MODEL_TYPE={MODEL_TYPE!r}, but the actual model object is {actual_model_type!r}. "
            "Saving under the actual model type."
        )

    file_stem = f"{actual_model_type}_{suffix}"
    model_path = MODEL_DIR / f"{file_stem}.pt"
    history_path = MODEL_DIR / f"{file_stem}_history.csv"
    metrics_path = MODEL_DIR / f"{file_stem}_metrics.json"

    torch.save({
        "model_state_dict": model.state_dict(),
        "model_type": actual_model_type,
        "model_class": type(model).__name__,
        "file_stem": file_stem,
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "feature_metadata": metadata,
        "label_names": label_names,
        "test_context_policy": data["test_context_policy"],
        "metrics": final_metrics,
        "use_tiny": bool(USE_TINY),
        "data_path": str(DATA_PATH),
    }, model_path)
    history.to_csv(history_path, index=False)

    import json
    metrics_path.write_text(json.dumps(final_metrics, indent=2) + "\n")

    print("saved model:", model_path)
    print("saved history:", history_path)
    print("saved metrics:", metrics_path)
else:
    print("SAVE_OUTPUTS is False; skipped saving.")


## Notes

- This notebook intentionally does not use legacy `edge_attr`.
- `x_cat` must go through embedding layers; do not concatenate raw categorical codes as continuous variables.
- Validation and test evaluation encode nodes from their allowed context graphs, not from the full query graph.
- The EdgeMLP can outperform the GNN if engineered edge/history features already capture most predictive signal or if graph smoothing is not helpful.
- Lower GNN validation/test AUC compared with a transductive graph model is expected when future graph edges are removed from message passing.
- The saved model is the best-validation checkpoint, not necessarily the final-iteration checkpoint.
